# Column Prepare — แก้ไข column ตามที่ column_audit.ipynb ตรวจพบ

เริ่มจากข้อ 1-3 ของ `column_audit.ipynb` (master column inventory, text variant groups,
quantitative format check) เอามาเป็นจุดตั้งต้น — โค้ดเหมือนกันทุกประการ (ยังเป็นขั้น
**ตรวจ/สำรวจ** อย่างเดียว ยังไม่แก้อะไร) แล้วค่อยต่อยอดเป็นขั้นตอนแก้ไขจริงใน cell ถัดๆ ไป

## Setup

In [1]:
import os, re, json
from collections import Counter
import numpy as np
import pandas as pd
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_rows", 200)

ROOT = os.path.abspath(os.path.join(os.getcwd(), ".." if os.path.basename(os.getcwd())=="notebooks" else "."))
os.chdir(ROOT)

TABLES = ["devices","layers","vendors","materials","process","properties","stability"]
t = {name: pd.read_csv(f"processed/normalize/{name}.csv") for name in TABLES}

# คอลัมน์ที่ "ตั้งใจให้เป็นตัวเลข" (audit แบบ format ตัวเลข ไม่ใช่ synonym)
QUANT_COLS = {"thickness_nm","amount","parameter","value","error","sample_no",
              "temperature_c","humidity_pct","time_hr","retention_pct","t80_hr",
              "initial_pce","final_pce"}
# id/โครงสร้าง ไม่ต้อง audit เนื้อหา
SKIP_COLS = {"device_id","source_file","source_sheet","layer_id","step_order","doi_raw"}

def kind(col):
    if col in QUANT_COLS: return "quant"
    return "text"

# normalizer สำหรับหา "ความหมายเดียวกันแต่เขียนต่าง" (ใช้กับ text เท่านั้น)
def tkey(v):
    s = str(v).strip().lower()
    s = re.sub(r"[\s\-_/():.,]+", "", s)   # ตัด separator/ช่องว่าง/เครื่องหมาย
    s = s.replace("−", "-")                # unicode minus -> ascii (ก่อนถูกตัดด้านบนแล้ว)
    return s

def variant_groups(series, keyfn=tkey):
    # คืนกลุ่มค่าที่ normalize แล้วตรงกัน แต่สะกดต่างกัน (>1 รูปแบบ)
    s = series.dropna().astype(str).map(str.strip)
    s = s[s != ""]
    groups = {}
    for v in s:
        groups.setdefault(keyfn(v), []).append(v)
    rows = []
    for k, vals in groups.items():
        c = Counter(vals)
        if len(c) > 1:
            rows.append({"canonical_key": k, "n_variants": len(c),
                         "variants": "  |  ".join(f"{x} ×{c[x]}" for x in sorted(c)),
                         "total_rows": sum(c.values())})
    out = pd.DataFrame(rows)
    return out.sort_values(["n_variants","total_rows"], ascending=False).reset_index(drop=True) if len(out) else out

print("โหลด", len(t), "ตาราง:", {k: v.shape for k, v in t.items()})

โหลด 7 ตาราง: {'devices': (44, 4), 'layers': (352, 5), 'vendors': (644, 4), 'materials': (543, 7), 'process': (702, 9), 'properties': (563, 6), 'stability': (62, 10)}


## 1. Master column inventory

In [2]:
def numeric_parse_rate(series):
    x = series.dropna().astype(str).str.strip()
    if len(x) == 0: return np.nan
    pure = x.str.fullmatch(r"-?\d+\.?\d*")
    return round(pure.mean(), 3)

rows = []
for name in TABLES:
    df = t[name]
    for col in df.columns:
        if col in SKIP_COLS: continue
        s = df[col]
        nn = s.notna().sum()
        info = {"table": name, "column": col, "kind": kind(col),
                "n_nonnull": int(nn), "pct_missing": round(s.isna().mean()*100, 1),
                "n_unique": int(s.dropna().astype(str).str.strip().nunique())}
        if kind(col) == "text":
            vg = variant_groups(s)
            info["variant_groups"] = 0 if len(vg)==0 else int(len(vg))
            info["numeric_parse_rate"] = np.nan
        else:
            info["variant_groups"] = np.nan
            info["numeric_parse_rate"] = numeric_parse_rate(s)
        rows.append(info)

inventory = pd.DataFrame(rows)
inventory

,table,column,kind,n_nonnull,pct_missing,n_unique,variant_groups,numeric_parse_rate
0,devices,sample_no,quant,44,0.0,5,NaN,1.000
1,layers,layer_type,text,349,0.9,12,2.0,NaN
2,layers,material,text,315,10.5,65,0.0,NaN
3,layers,thickness_nm,quant,105,70.2,25,NaN,0.886
4,vendors,material,text,644,0.0,79,0.0,NaN
5,vendors,purity,text,346,46.3,22,0.0,NaN
6,vendors,vendor,text,562,12.7,59,2.0,NaN
7,materials,material,text,511,5.9,61,0.0,NaN
8,materials,type,text,375,30.9,22,1.0,NaN
9,materials,solvent,text,337,37.9,37,6.0,NaN


## 2. Text columns — ค่าที่ "ความหมายเดียวกันแต่เขียนต่าง"

In [3]:
allvg = []
for name in TABLES:
    df = t[name]
    for col in df.columns:
        if col in SKIP_COLS or kind(col) != "text": continue
        vg = variant_groups(df[col])
        if len(vg):
            vg.insert(0, "table", name); vg.insert(1, "column", col)
            allvg.append(vg)
allvg = pd.concat(allvg, ignore_index=True).sort_values(
    ["n_variants","total_rows"], ascending=False).reset_index(drop=True)
print(f"พบ {len(allvg)} กลุ่ม ที่ความหมายเดียวกันแต่เขียนต่างกัน\n")
allvg[["table","column","n_variants","total_rows","variants"]]

พบ 35 กลุ่ม ที่ความหมายเดียวกันแต่เขียนต่างกัน



,table,column,n_variants,total_rows,variants
0,materials,solvent,4,114,DMF/DMSO (4/1 V/V) ×51 | DMF/DMSO (4:1 V/V) ×43 | DMF/DMSO (4:1 v/v) ×8 | DMF:DMSO (4:1 v/v) ×12
1,properties,unit,4,69,mA cm-2 ×52 | mA/cm-2 ×2 | mA/cm2 ×3 | mAcm-2 ×12
2,materials,solvent,4,6,DMF /DMSO (3.08/1 V/V) ×1 | DMF/DMSO (3.08/1 V/V) ×1 | DMF/DMSO (3.08/1 v/v) ×2 | DMFDMSO (3.08/1 v/v) ×2
3,process,process_type,3,82,Thermal Evaporation ×12 | Thermal evaporation ×6 | ThermalEvaporation ×64
4,process,process_type,2,213,Spin coating ×9 | SpinCoating ×204
5,process,unit,2,157,"C, sec ×153 | C,sec ×4"
6,vendors,vendor,2,100,Sigma Aldrich ×12 | Sigma-Aldrich ×88
7,layers,layer_type,2,43,HTL Treatment ×1 | HTL treatment ×42
8,process,process_type,2,42,AntiSolvent ×35 | Antisolvent ×7
9,layers,layer_type,2,40,PSK Treatment ×1 | PSK treatment ×39


## 3. Quantitative columns — format check

คอลัมน์ที่ควรเป็นตัวเลขแต่เก็บเป็น string: ดูว่ากี่ % เป็นเลขล้วน, กี่ % ปนตัวอักษร/หน่วย,
เป็นช่วง (`a - b`), หรือหลายค่าในช่องเดียว (`80,12`).

In [4]:
def quant_profile(series):
    x = series.dropna().astype(str).map(str.strip)
    x = x[x != ""]
    n = len(x)
    if n == 0: return None
    pure   = x.str.fullmatch(r"-?\d+\.?\d*")
    has_a  = x.str.contains(r"[A-Za-zΩµμÅ°%]")
    rng    = x.str.contains(r"\d\s*[-–~]\s*\d")
    multi  = x.str.contains(r",|/| x ", regex=True)
    return {"n": n,
            "pure_number_%": round(pure.mean()*100,1),
            "has_letters/unit_%": round(has_a.mean()*100,1),
            "range_%": round(rng.mean()*100,1),
            "multivalue_%": round(multi.mean()*100,1),
            "examples_dirty": " | ".join(x[~pure].unique()[:5])}

rows = []
for name in TABLES:
    df = t[name]
    for col in df.columns:
        if col in SKIP_COLS or kind(col) != "quant": continue
        p = quant_profile(df[col])
        if p: rows.append({"table": name, "column": col, **p})
quant_report = pd.DataFrame(rows)
quant_report

,table,column,n,pure_number_%,has_letters/unit_%,range_%,multivalue_%,examples_dirty
0,devices,sample_no,44,100.0,0.0,0.0,0.0,
1,layers,thickness_nm,105,88.6,5.7,3.8,1.9,"400-450 | 4, 100 | 400 Å | 90 Å | 1200 Å"
2,materials,amount,323,100.0,0.0,0.0,0.0,
3,process,parameter,595,26.4,4.2,1.7,69.4,"3000, 30 | 270, 10800 | 60, 3600 | 1000,500,10 | 4000,2000,35"
4,properties,value,544,96.9,3.1,0.0,0.0,3.3e-08 | 3.06e-08 | 2.84e-08 | 8.1e+18 | 1.51e+18
5,properties,error,74,100.0,0.0,0.0,0.0,
6,stability,temperature_c,48,87.5,12.5,0.0,0.0,RT
7,stability,humidity_pct,26,53.8,0.0,26.9,0.0,50 - 60 | 50–60 | 45–55 | 30-50 | 60 ± 5
8,stability,time_hr,59,100.0,0.0,0.0,0.0,
9,stability,retention_pct,50,98.0,0.0,0.0,0.0,< 55


## 4-6. ต่อจาก column_audit.ipynb — เฉพาะส่วนที่ยังไม่ถูกครอบคลุมโดย PART 1/2 ด้านล่าง

ข้าม 4.2 (layer/materials.type), 4.4 (solvent), 4.5 (unit spelling), 4.6 (vendor) — คอลัมน์พวกนี้อยู่ใน `TEXT_COLS` ของ PART 1 อยู่แล้ว (variant-merge คลุมให้แล้ว) และข้าม 4.7 (แตกชื่อ property) เพราะ `eda_prepare.ipynb` PART 3 ทำอยู่แล้วด้วย logic ที่ละเอียดกว่า (`decompose()`) — ทำซ้ำจะสับสน/เสี่ยงขัดแย้งกัน

ที่เหลือ (4.1, 4.3, 4.8, 5, 6) ยังเป็นแค่**ตรวจ**อย่างเดียวเหมือนเดิม (ยังไม่มีที่ไหน apply เป็น fix จริง) — โดยเฉพาะ §5.3 (Voc/Jsc ติด unit ผิด, mobility หน่วยผิด) เป็นช่องโหว่จริงที่ pipeline ปัจจุบันยังไม่ได้แก้

### 4.1 `process.process_type` — ขั้นตอนการผลิต (จุดเด่นของโปรเจกต์)

In [5]:
PROCESS_CANON = {
 "spincoating":"spin_coating", "spin coating":"spin_coating",
 "annealing":"annealing", "postannealing":"post_annealing",
 "thermalevaporation":"thermal_evaporation", "thermal evaporation":"thermal_evaporation",
 "electronbeamevaporation":"ebeam_evaporation",
 "antisolvent":"antisolvent",
 "wash":"wash", "ethanolwash":"wash", "dry":"dry",
 "plasmatreat":"plasma_treatment", "plasma treat":"plasma_treatment",
 "uv/ozone treat":"uv_ozone_treatment", "uv treat":"uv_treatment", "treatment":"treatment",
 "stir":"stir", "magnetic stir":"stir",
 "mix":"mixing", "mixing":"mixing",
 "sonication":"sonication", "dip-coating":"dip_coating", "dissolve":"dissolve",
 "ph adjustment":"ph_adjustment",
 "sputtering":"sputtering", "rfsputtering":"sputtering", "magnetronsputtering":"sputtering",
 "presputtering":"pre_sputtering", "pre-sputtering":"pre_sputtering",
 "ald":"ald",
}
def map_report(series, canon):
    vc = series.dropna().astype(str).str.strip().value_counts()
    df = vc.rename_axis("raw").reset_index(name="count")
    df["canonical"] = df["raw"].str.lower().map(canon)
    return df

pt = map_report(t["process"]["process_type"], PROCESS_CANON)
unmapped = pt[pt["canonical"].isna()]
print(f"raw {pt['raw'].nunique()} แบบ -> canonical {pt['canonical'].nunique()} แบบ | "
      f"map ไม่เจอ {len(unmapped)} แบบ ({unmapped['count'].sum()} rows)")
pt

raw 32 แบบ -> canonical 21 แบบ | map ไม่เจอ 0 แบบ (0 rows)


,raw,count,canonical
0,SpinCoating,204,spin_coating
1,Annealing,161,annealing
2,ThermalEvaporation,64,thermal_evaporation
3,Wash,52,wash
4,AntiSolvent,35,antisolvent
5,Dry,32,dry
6,Plasma Treat,17,plasma_treatment
7,UV/Ozone Treat,14,uv_ozone_treatment
8,Thermal Evaporation,12,thermal_evaporation
9,Spin coating,9,spin_coating


### 4.3 ชื่อวัสดุ (`material`) — จับกลุ่ม chemistry family

ชื่อวัสดุปรากฏใน 3 ตาราง (`layers`, `materials`, `vendors`). ตรวจว่า variant เดียวกันไหม และจับตระกูล เช่น `NiOx / NiOx NPs / NiOx Nanoparticles` = NiOx.

In [6]:
def material_family(x):
    s = str(x).lower()
    if "nio" in s: return "NiOx"
    if s.startswith("c60") or s=="c60": return "C60"
    if "bcp" in s: return "BCP"
    if "pbi2" in s: return "PbI2"
    if "pedot" in s: return "PEDOT:PSS"
    if "spiro" in s: return "Spiro-OMeTAD"
    if any(k in s for k in ["pacz","2pacz","me-2pacz","mecz"]): return "SAM (xPACz)"
    return None

mats = pd.concat([t["layers"][["material"]].assign(src="layers"),
                  t["materials"][["material"]].assign(src="materials"),
                  t["vendors"][["material"]].assign(src="vendors")], ignore_index=True)
mats = mats.dropna()
mats["material"] = mats["material"].astype(str).str.strip()
mats["family"] = mats["material"].map(material_family)
fam = (mats.dropna(subset=["family"]).groupby("family")["material"]
          .agg(lambda s: "  |  ".join(f"{v} ×{c}" for v,c in Counter(s).most_common()))
          .reset_index(name="spellings_found"))
fam["n_spellings"] = fam["spellings_found"].str.count(r"\|") + 1
print("ตระกูลวัสดุที่มีหลายชื่อ:")
fam.sort_values("n_spellings", ascending=False)

ตระกูลวัสดุที่มีหลายชื่อ:


,family,spellings_found,n_spellings
2,NiOx,NiOx ×51 | NiOx NPs ×35 | NiO target ×6 | NiOx Nanoparticles ×5 | NiOx NPs ink ×2 | Cu:NiO ×1 | NiOx:TTT...,9
4,SAM (xPACz),Me-4PACz ×10 | 2PACz ×4 | Me-2PACz ×4 | Me-2PACz (SAM) ×2 | Me-4PACz + PFPTFB ×1 | 2PACz SAM ×1 | Me-4PA...,9
0,BCP,"BCP ×111 | PbI2, BCP ×5 | BCP (bathocuproine) ×2 | BCP + SnO2 ×1",4
1,C60,C60 ×52,1
3,PbI2,PbI2 ×78,1


### 4.8 `stability.environment` — จับกลุ่มบรรยากาศ/โปรโตคอล

ปนกันหลายมิติ: บรรยากาศ (N2/Air), โปรโตคอล ISOS, โหมดวัด (MPP/MPPT). เสนอ `atmosphere`.

In [7]:
def atmosphere(x):
    s = str(x).lower()
    if "n2" in s or "nitrogen" in s: return "N2/inert"
    if "ambient" in s or "air" in s or "humidity" in s: return "ambient/air"
    if "thermal" in s: return "thermal"
    if "mpp" in s or "operational" in s: return "operational(MPP)"
    if "isos" in s: return "ISOS (unspecified atm.)"
    return "other"
env = t["stability"]["environment"].dropna().str.strip()
edf = env.value_counts().rename_axis("raw").reset_index(name="count")
edf["atmosphere"] = edf["raw"].map(atmosphere)
edf.sort_values(["atmosphere","count"], ascending=[True,False])

,raw,count,atmosphere
2,ISOS-L2,4,ISOS (unspecified atm.)
19,ISOS-L-3,1,ISOS (unspecified atm.)
0,N2,11,N2/inert
3,N2/ISOS-L-2I,4,N2/inert
4,Nitrogen (N2),3,N2/inert
8,N2/ISOS-D-2,3,N2/inert
12,N2 atmosphere,2,N2/inert
15,"N2 glovebox, AM1.5G 1-sun illumination",2,N2/inert
18,N2 storage,1,N2/inert
22,N2/ISOS-L-2I (with top passivation),1,N2/inert


## 5. Unit audit — เช็คหน่วยทุกคอลัมน์

ตรวจ 3 ระดับ:
1. **หน่วยฝังในชื่อ header** (`thickness_nm`, `temperature_c`, …) — ค่าอยู่ในช่วงที่หน่วยนั้นสมเหตุผลไหม
2. **ปริมาณเดียวกันใน `properties`** — หน่วยตรงกันทุกแถวไหม
3. **ค่าที่ติดหน่วยผิด** + **micro-sign vs greek-mu** ที่ตาเปล่ามองไม่เห็น

### 5.1 หน่วยฝังในชื่อ header — ค่าอยู่ในช่วงสมเหตุผลไหม

In [8]:
hdr_units = {
 ("layers","thickness_nm","nm"):(0,2000),
 ("stability","temperature_c","C"):(-20,700),
 ("stability","humidity_pct","%"):(0,100),
 ("stability","time_hr","hr"):(0,50000),
 ("stability","retention_pct","%"):(0,110),
 ("stability","initial_pce","%"):(0,30),
 ("stability","final_pce","%"):(0,30),
}
rows = []
for (tab,col,unit),(lo,hi) in hdr_units.items():
    s = pd.to_numeric(t[tab][col], errors="coerce").dropna()
    oob = s[(s<lo)|(s>hi)]
    rows.append({"table":tab,"column":col,"unit":unit,"n":int(len(s)),
                 "min":round(float(s.min()),3),"max":round(float(s.max()),3),
                 "expected_range":f"[{lo},{hi}]","n_out_of_range":int(len(oob)),
                 "status":"OK" if len(oob)==0 else "CHECK"})
unit_header_check = pd.DataFrame(rows)
unit_header_check

,table,column,unit,n,min,max,expected_range,n_out_of_range,status
0,layers,thickness_nm,nm,93,5.00,715.00,"[0,2000]",0,OK
1,stability,temperature_c,C,42,25.00,85.00,"[-20,700]",0,OK
2,stability,humidity_pct,%,14,0.00,60.00,"[0,100]",0,OK
3,stability,time_hr,hr,59,0.83,5000.00,"[0,50000]",0,OK
4,stability,retention_pct,%,49,0.40,100.85,"[0,110]",0,OK
5,stability,initial_pce,%,4,16.25,17.11,"[0,30]",0,OK
6,stability,final_pce,%,4,10.50,14.50,"[0,30]",0,OK


### 5.2 `properties` — ปริมาณเดียวกัน หน่วยตรงกันทุกแถวไหม

In [9]:
def metric_of(name):
    n = str(name).lower()
    for k,b in [("pce","PCE"),("voc","Voc"),("jsc","Jsc"),("ff","FF"),
                ("band","bandgap"),("vbm","VBM"),("fermi","Fermi level"),
                ("work function","Work function"),("mobility","mobility"),
                ("trap","trap density"),("grain","grain size"),
                ("contact angle","contact angle"),("rms","roughness RMS"),
                ("conductivity","conductivity"),("lifetime","carrier lifetime"),
                ("eqe","EQE"),("spo","SPO")]:
        if k in n: return b
    if re.search(r"\brs\b", n) or "series resist" in n: return "series R (Rs)"
    if "rsh" in n or "shunt" in n: return "shunt R (Rsh)"
    return "other"

pp = t["properties"].copy()
pp["metric"] = pp["property"].map(metric_of)
pp["u"] = pp["unit"].astype(str).str.strip()
rows = []
for m, grp in pp.dropna(subset=["unit"]).groupby("metric"):
    uv = grp["u"].value_counts()
    rows.append({"metric":m,"n":int(uv.sum()),"n_distinct_units":int(uv.size),
                 "units":", ".join(f"{u} x{c}" for u,c in uv.items()),
                 "flag":"MULTI-UNIT" if uv.size>1 else "OK"})
prop_unit_consistency = (pd.DataFrame(rows)
    .sort_values(["flag","n"], ascending=[True,False]).reset_index(drop=True))
prop_unit_consistency

,metric,n,n_distinct_units,units,flag
0,other,101,12,"V x35, ns x12, % x12, eV x11, Ω x8, nm x6, Ohm x5, cm-3 x4, F x3, Ω cm² x3, cm-3 eV-1 x1, cm-3 e/V x1",MULTI-UNIT
1,Jsc,79,6,"mA cm-2 x52, mAcm-2 x12, % x6, mA cm−2 x4, mA/cm2 x3, mA/cm-2 x2",MULTI-UNIT
2,Voc,67,2,"V x61, % x6",MULTI-UNIT
3,grain size,27,2,"nm x26, μm x1",MULTI-UNIT
4,series R (Rs),23,6,"Ohm x6, Ω x6, Ohm/cm2 x4, Ω cm² x3, Ω/sq x2, Ω cm2 x2",MULTI-UNIT
5,contact angle,18,2,"deg x14, degrees x4",MULTI-UNIT
6,mobility,9,3,"cm2 V-1 s-1 x4, m2 V-1 s-1 x3, cm2/Vs x2",MULTI-UNIT
7,PCE,73,1,% x73,OK
8,FF,59,1,% x59,OK
9,trap density,26,1,cm-3 x26,OK


### 5.3 ค่าที่ติดหน่วยผิด (value-level)

- **Voc/Jsc ที่ unit = `%`** → ควรเป็น `V` / `mA·cm⁻²`
- **mobility** → ปน `m²` กับ `cm²` (ต่างกัน 10⁴ เท่า)
- **FF** → ปน fraction (<1) กับ percent

In [10]:
mislabel = pp[(pp["u"]=="%") & (pp["metric"].isin(["Voc","Jsc"]))][["device_id","property","value","unit"]]
print(f"[Voc/Jsc ติด unit '%'] {len(mislabel)} แถว (ควรเป็น V / mA cm-2):")
display(mislabel)

mob = pp[pp["metric"]=="mobility"][["device_id","property","value","unit"]]
mob_m2 = mob[mob["unit"].astype(str).str.contains("m2 V", na=False)]
print(f"[mobility] {len(mob)} แถว, ในนั้นเป็น m2 (ต้อง x1e4 -> cm2) {len(mob_m2)} แถว:")
display(mob)

ffm = pp[pp["metric"]=="FF"].copy()
ffm["v"] = pd.to_numeric(ffm["value"], errors="coerce")
ff_frac = ffm[ffm["v"]<1][["device_id","property","value","unit"]]
print(f"[FF] fraction(<1)={int((ffm['v']<1).sum())}  percent(>=1)={int((ffm['v']>=1).sum())}:")
display(ff_frac)

[Voc/Jsc ติด unit '%'] 12 แถว (ควรเป็น V / mA cm-2):


,device_id,property,value,unit
48,10.1016j.jechem.2021.11.029_Sample1,Voc (RS),1.064,%
49,10.1016j.jechem.2021.11.029_Sample1,Jsc (RS),21.320,%
52,10.1016j.jechem.2021.11.029_Sample1,Voc (FS),1.061,%
53,10.1016j.jechem.2021.11.029_Sample1,Jsc (FS),21.120,%
56,10.1016j.jechem.2021.11.029_Sample1,Average Voc,1.065,%
57,10.1016j.jechem.2021.11.029_Sample1,Average Jsc,20.780,%
68,10.1016j.jechem.2021.11.029_Sample2,Voc (RS),1.118,%
69,10.1016j.jechem.2021.11.029_Sample2,Jsc (RS),21.980,%
72,10.1016j.jechem.2021.11.029_Sample2,Voc (FS),1.122,%
73,10.1016j.jechem.2021.11.029_Sample2,Jsc (FS),22.120,%


[mobility] 9 แถว, ในนั้นเป็น m2 (ต้อง x1e4 -> cm2) 7 แถว:


,device_id,property,value,unit
65,10.1016j.jechem.2021.11.029_Sample1,Hole Mobility,2.970000e-04,cm2/Vs
86,10.1016j.jechem.2021.11.029_Sample2,Hole Mobility,2.660000e-04,cm2/Vs
123,10.1021_acsami.5c16972_Sample1,Hole mobility,3.671000e-04,m2 V-1 s-1
143,10.1021_acsami.5c16972_Sample2,Hole mobility,4.646000e-04,m2 V-1 s-1
163,10.1021_acsami.5c16972_Sample3,Hole mobility,6.941000e-04,m2 V-1 s-1
189,10_1002_smll_202504237_Sample1,Hole mobility,3.460000e+00,cm2 V-1 s-1
200,10_1002_smll_202504237_Sample2,Hole mobility,4.320000e+01,cm2 V-1 s-1
221,10_1002_solr_202200422_Sample1,Hole mobility,8.300000e-07,cm2 V-1 s-1
242,10_1002_solr_202200422_Sample2,Hole mobility,2.300000e-06,cm2 V-1 s-1


[FF] fraction(<1)=8  percent(>=1)=59:


,device_id,property,value,unit
2,10.1002admi.202500231_Sample1,FF,0.76,NaN
14,10.1002admi.202500231_Sample2,FF,0.75,NaN
26,10.1002admi.202500231_Sample3,FF,0.78,NaN
38,10.1002admi.202500231_Sample4,FF,0.79,NaN
349,10_1016_j_solener_2025_113286_Sample1,Average FF,0.69,NaN
353,10_1016_j_solener_2025_113286_Sample1,Champion FF,0.71,NaN
364,10_1016_j_solener_2025_113286_Sample2,Average FF,0.72,NaN
368,10_1016_j_solener_2025_113286_Sample2,Champion FF,0.72,NaN


### 5.4 micro-sign `µ` (U+00B5) vs greek-mu `μ` (U+03BC)

หน้าตาเหมือนกันเป๊ะแต่คนละ codepoint → exact-match แยกเป็นคนละหน่วยทั้งที่เป็น µL เดียวกัน
(อ่านจาก `processed/normalize/` สดๆ ตั้งใจ — เพื่อดูสภาพ**ก่อน** PART 1 unicode-fix ด้านล่างจะรวมให้)

In [11]:
MICRO, MU = chr(0x00B5), chr(0x03BC)
rows = []
for tab in ["materials","process","properties"]:
    for val in pd.read_csv(f"processed/normalize/{tab}.csv")["unit"].dropna().astype(str).unique():
        if MICRO in val or MU in val:
            rows.append({"table":f"{tab}.unit","raw_unit":val,
                         "micro_sign_U+00B5": MICRO in val,
                         "greek_mu_U+03BC": MU in val,
                         "codepoints":" ".join(f"U+{ord(c):04X}" for c in val if ord(c)>127)})
micro_check = pd.DataFrame(rows)
micro_check

,table,raw_unit,micro_sign_U+00B5,greek_mu_U+03BC,codepoints
0,materials.unit,μL,False,True,U+03BC
1,materials.unit,µL,True,False,U+00B5
2,materials.unit,µl,True,False,U+00B5
3,process.unit,"µL, sec",True,False,U+00B5
4,process.unit,"μL, sec",False,True,U+03BC
5,process.unit,μL,False,True,U+03BC
6,process.unit,"µl, sec",True,False,U+00B5
7,process.unit,µL,True,False,U+00B5
8,properties.unit,μm,False,True,U+03BC


### 5.5 Export unit reports

In [12]:
os.makedirs("processed/audit", exist_ok=True)
prop_unit_consistency.to_csv("processed/audit/unit_consistency_properties.csv", index=False)
unit_value_errors = pd.concat([
    mislabel.assign(issue="Voc/Jsc labeled %"),
    mob_m2.assign(issue="mobility in m2 (needs x1e4)"),
    ff_frac.assign(issue="FF as fraction (needs x100)"),
], ignore_index=True)
unit_value_errors.to_csv("processed/audit/unit_value_errors.csv", index=False)
micro_check.to_csv("processed/audit/unit_micro_sign.csv", index=False)
print("wrote -> processed/audit/{unit_consistency_properties, unit_value_errors, unit_micro_sign}.csv")
print("value-level unit errors:", len(unit_value_errors), "rows")

wrote -> processed/audit/{unit_consistency_properties, unit_value_errors, unit_micro_sign}.csv
value-level unit errors: 27 rows


## 6. Cross-table consistency — ชื่อวัสดุข้ามตารางตรงกันไหม

วัสดุตัวเดียวกันควรสะกดเหมือนกันใน `layers`, `materials`, `vendors`.
หา normalized-key ที่มีหลายสะกด "ข้ามตาราง".

In [13]:
frames = []
for name in ["layers","materials","vendors"]:
    s = t[name]["material"].dropna().astype(str).str.strip()
    frames.append(pd.DataFrame({"table": name, "material": s, "key": s.map(tkey)}))
allm = pd.concat(frames, ignore_index=True)
rows = []
for k, g in allm.groupby("key"):
    spell = sorted(g["material"].unique())
    if len(spell) > 1:
        rows.append({"key": k, "n_spellings": len(spell),
                     "spellings": "  |  ".join(spell),
                     "tables": ", ".join(sorted(g["table"].unique()))})
cross = pd.DataFrame(rows, columns=["key","n_spellings","spellings","tables"])
if len(cross):
    cross = cross.sort_values("n_spellings", ascending=False).reset_index(drop=True)
print(f"{len(cross)} วัสดุที่สะกดไม่ตรงกัน (normalize แล้ว key เดียวกันแต่หลายสะกด)")
cross if len(cross) else "✅ ชื่อวัสดุสะกดสม่ำเสมอ (ภายใต้ normalize) ทั้งใน layers/materials/vendors"

0 วัสดุที่สะกดไม่ตรงกัน (normalize แล้ว key เดียวกันแต่หลายสะกด)


'✅ ชื่อวัสดุสะกดสม่ำเสมอ (ภายใต้ normalize) ทั้งใน layers/materials/vendors'

# PART 1 · Clean dirty text

พอร์ตมาจาก `eda_prepare.ipynb` PART 1 เดิมทีเดียว (2026-07-15) — โค้ดเหมือนทุกประการ เป็นขั้นแรกของ `column_prepare.ipynb` แทนที่ `eda_prepare.ipynb` จะไม่ทำ PART 1/2 ซ้ำอีก (PART 3 ของ eda_prepare จะอ่านต่อจาก `processed/clean/` ที่เขียนออกท้ายนี้แทน).

In [14]:
import re
from collections import Counter
from pathlib import Path

import pandas as pd

ROOT = Path.cwd() if (Path.cwd() / "processed" / "normalize").exists() else Path.cwd().parent
SRC = ROOT / "processed" / "normalize"
AUDIT = ROOT / "processed" / "audit"

# dtype=str -> pandas ไม่แปลงตัวเลข (100 -> 100.0) เขียนกลับได้ตรงต้นฉบับ
tables = {p.stem: pd.read_csv(p, dtype=str) for p in sorted(SRC.glob("*.csv"))}
tables_raw = {name: df.copy() for name, df in tables.items()}   # สำเนา "ก่อน clean" ไว้เทียบตอนจบ

diff_log = []   # ทุกเซลล์ที่ถูกแก้: step, table, column, row, device_id, before, after

def log_and_apply(name: str, col: str, new: pd.Series, step: str) -> int:
    """เทียบคอลัมน์เดิมกับค่าใหม่ บันทึกเซลล์ที่ต่างลง diff_log แล้วค่อย apply"""
    df = tables[name]
    old = df[col]
    mask = old.notna() & (old != new)
    for idx in df.index[mask]:
        diff_log.append({"step": step, "table": name, "column": col, "row": int(idx),
                         "device_id": df.at[idx, "device_id"] if "device_id" in df.columns else "",
                         "before": old.at[idx], "after": new.at[idx]})
    # .astype(object) กันคอลัมน์ที่ว่างทั้งคอลัมน์ dtype เพี้ยนเป็น float หลัง .map()
    df[col] = new.astype(object)
    return int(mask.sum())

def step_diff(step: str) -> pd.DataFrame:
    """สรุป diff ของขั้นนั้นเป็นคู่ (before -> after) ไม่ซ้ำ พร้อมจำนวนเซลล์"""
    d = pd.DataFrame([r for r in diff_log if r["step"] == step],
                     columns=["step", "table", "column", "row", "device_id", "before", "after"])
    return (d.groupby(["table", "column", "before", "after"])
              .size().rename("n_cells").reset_index())

print("โหลด", len(tables), "ตาราง:", {k: v.shape for k, v in tables.items()})

โหลด 7 ตาราง: {'devices': (44, 4), 'layers': (352, 5), 'materials': (543, 7), 'process': (702, 9), 'properties': (563, 6), 'stability': (62, 10), 'vendors': (644, 4)}


## 1.1 Unicode/whitespace normalize — ทุกคอลัมน์ของทุกตาราง

In [15]:
UNICODE_FIX = str.maketrans({"μ": "µ",   # Greek mu -> micro sign
                             "−": "-",          # Unicode minus -> hyphen
                             " ": " "})         # NBSP -> space

def norm_text(s: str) -> str:
    return re.sub(r"\s+", " ", s.translate(UNICODE_FIX)).strip()

n_fixed = 0
for name, df in tables.items():
    for col in df.columns:
        after = df[col].map(lambda v: norm_text(v) if isinstance(v, str) else v)
        n_fixed += log_and_apply(name, col, after, "1-unicode/ws")
print("1-unicode/ws: แก้", n_fixed, "เซลล์")

1-unicode/ws: แก้ 25 เซลล์


## 1.2 Typo fixes ที่รู้ตัวแล้ว

In [16]:
TYPO_FIX = {
    ("properties", "property"): {
        "Trap densitiy": "Trap density",  # สะกดผิด
        "AverageVoc": "Average Voc",      # space หาย
    },
}

n_fixed = 0
for (name, col), fixes in TYPO_FIX.items():
    n_fixed += log_and_apply(name, col, tables[name][col].replace(fixes), "2-typo")
print("2-typo: แก้", n_fixed, "เซลล์")

2-typo: แก้ 4 เซลล์


## 1.3 Variant-merge อัตโนมัติ (case/วรรคตอนต่างกัน) — majority vote

In [17]:
TEXT_COLS = {
    "layers":     ["layer_type", "material"],
    "materials":  ["material", "type", "solvent", "unit"],
    "process":    ["process_type", "description", "unit"],
    "properties": ["property", "unit"],
    "stability":  ["environment", "encapsulation"],
    "vendors":    ["material", "vendor"],
}

# ตัวสะกดมาตรฐานที่เลือกเอง (ถ้าไม่กำหนด ใช้ majority vote)
PREFERRED = {
    ("process", "process_type", "spincoating"): "Spin coating",
    ("process", "process_type", "thermalevaporation"): "Thermal evaporation",
    ("materials", "solvent", "dmfdmso41vv"): "DMF/DMSO (4:1 v/v)",
    ("materials", "solvent", "dmfdmso3081vv"): "DMF/DMSO (3.08/1 v/v)",
    ("materials", "solvent", "dmfdmso41"): "DMF/DMSO (4:1)",
    ("properties", "property", "grainsize"): "Grain size",  # เสมอกัน 10:10
}

def variant_key(s: str) -> str:
    # \W ตัด punctuation แต่เก็บตัวอักษร unicode ไว้ (เช่น µ) — กัน 'µL' ไปชนกับ 'L'
    return re.sub(r"[\W_]", "", s.lower())

map_rows = []
for name, cols in TEXT_COLS.items():
    for col in cols:
        counts = Counter(tables[name][col].dropna())
        groups = {}
        for val, n in counts.items():
            k = variant_key(val)
            if re.search(r"[^\W\d_]", k):          # ต้องมีตัวอักษร
                groups.setdefault(k, []).append((val, n))
        for k, variants in groups.items():
            if len(variants) < 2 and (name, col, k) not in PREFERRED:
                continue
            canonical = PREFERRED.get((name, col, k)) or \
                        max(variants, key=lambda v: (v[1], v[0]))[0]
            for val, n in variants:
                if val != canonical:
                    map_rows.append({"table": name, "column": col,
                                     "raw": val, "canonical": canonical, "n_rows": n,
                                     "rule": "override" if (name, col, k) in PREFERRED
                                             else "majority"})

var_map = (pd.DataFrame(map_rows,
                        columns=["table", "column", "raw", "canonical", "n_rows", "rule"])
             .sort_values(["table", "column", "canonical", "raw"])
             .reset_index(drop=True))
var_map

,table,column,raw,canonical,n_rows,rule
0,layers,layer_type,HTL Treatment,HTL treatment,1,majority
1,layers,layer_type,PSK Treatment,PSK treatment,1,majority
2,materials,solvent,DMF /DMSO (3.08/1 V/V),DMF/DMSO (3.08/1 v/v),1,override
3,materials,solvent,DMF/DMSO (3.08/1 V/V),DMF/DMSO (3.08/1 v/v),1,override
4,materials,solvent,DMFDMSO (3.08/1 v/v),DMF/DMSO (3.08/1 v/v),2,override
5,materials,solvent,DMF/DMSO (4/1 V/V),DMF/DMSO (4:1 v/v),51,override
6,materials,solvent,DMF/DMSO (4:1 V/V),DMF/DMSO (4:1 v/v),43,override
7,materials,solvent,DMF:DMSO (4:1 v/v),DMF/DMSO (4:1 v/v),12,override
8,materials,solvent,DMF:DMSO (4:1),DMF/DMSO (4:1),10,override
9,materials,solvent,De-ionized water,Deionized water,3,majority


## 1.4 Apply variant-merge mapping

In [18]:
report = []
for (name, col), g in var_map.groupby(["table", "column"]):
    mapping = dict(zip(g["raw"], g["canonical"]))
    before_unique = tables[name][col].nunique()
    n = log_and_apply(name, col, tables[name][col].replace(mapping), "3-variant")
    after_unique = tables[name][col].nunique()
    report.append({"table": name, "column": col, "n_cells": n,
                   "n_unique_before": before_unique, "n_unique_after": after_unique})
pd.DataFrame(report)

,table,column,n_cells,n_unique_before,n_unique_after
0,layers,layer_type,2,12,10
1,materials,solvent,126,37,27
2,materials,type,2,22,21
3,materials,unit,14,21,18
4,process,description,3,47,44
5,process,process_type,293,32,26
6,process,unit,15,33,28
7,properties,property,30,97,91
8,properties,unit,17,26,23
9,stability,environment,2,23,22


## 1.5 สรุป diff + audit อิสระ (ไม่พึ่ง log)

In [19]:
diff_cells = pd.DataFrame(diff_log,
                          columns=["step", "table", "column", "row", "device_id",
                                   "before", "after"])

# นับอิสระจากสำเนาก่อน clean (ไม่พึ่ง log)
recount = sum(int((tables_raw[n][c].notna() & (tables_raw[n][c] != tables[n][c])).sum())
              for n in tables for c in tables[n].columns)
n_cells_logged = len(diff_cells[["table", "column", "row"]].drop_duplicates())
ok = "ตรงกัน ✓" if recount == n_cells_logged else "ไม่ตรง — ต้องตรวจ!"
print(f"เซลล์ที่แก้ (นับจาก log)      : {n_cells_logged}")
print(f"เซลล์ที่แก้ (นับอิสระจาก raw) : {recount}  ({ok})")

เซลล์ที่แก้ (นับจาก log)      : 566
เซลล์ที่แก้ (นับอิสระจาก raw) : 566  (ตรงกัน ✓)


## 1.6 Save audits

In [20]:
AUDIT.mkdir(parents=True, exist_ok=True)
var_map.to_csv(AUDIT / "text_cleanup_map.csv", index=False)
diff_cells.to_csv(AUDIT / "clean_diff_cells.csv", index=False)
print("wrote", AUDIT / "text_cleanup_map.csv")
print("wrote", AUDIT / "clean_diff_cells.csv")

wrote D:\University\Year 4 Semester 1 (Intern)\Perovskites-Research\processed\audit\text_cleanup_map.csv
wrote D:\University\Year 4 Semester 1 (Intern)\Perovskites-Research\processed\audit\clean_diff_cells.csv


## 1.7 Verify — self-check ก่อนไป PART 2

In [21]:
leftover = []
for name, cols in TEXT_COLS.items():
    for col in cols:
        groups = {}
        for val in tables[name][col].dropna():
            k = variant_key(val)
            if re.search(r"[^\W\d_]", k):
                groups.setdefault(k, set()).add(val)
        leftover += [(name, col, sorted(v)) for v in groups.values() if len(v) > 1]

bad_chars = sum(int(tables[n][c].fillna("").str.contains("[μ− ]").sum())
                for n in tables for c in tables[n].columns)
same_shape = all(tables[n].shape == tables_raw[n].shape for n in tables)

print("1) ไม่มี variant กลุ่มไหนหลงเหลือ :", not leftover)
print("2) ไม่มีอักขระ mu/minus/NBSP หลงเหลือ :", bad_chars == 0)
print("3) shape ตารางไม่เปลี่ยน :", same_shape)
assert not leftover and bad_chars == 0 and same_shape, "VERIFY FAILED"
print("PART 1 verify ผ่านหมด ✓")

1) ไม่มี variant กลุ่มไหนหลงเหลือ : True
2) ไม่มีอักขระ mu/minus/NBSP หลงเหลือ : True
3) shape ตารางไม่เปลี่ยน : True
PART 1 verify ผ่านหมด ✓


# PART 2 · Clean value formats

ต่อจาก PART 1 — ใช้ตาราง `tables` ใน memory ต่อเลย (ไม่มีโฟลเดอร์กลาง). กติกาต่างจาก PART 1: ไม่ทับคอลัมน์เดิม — เพิ่มคอลัมน์ใหม่ `_parsed`/`_note` แทนเสมอ

In [22]:
# ต่อจาก PART 1: ใช้ตาราง `tables` ใน memory ต่อเลย (ไม่มีโฟลเดอร์กลาง)
tables_after_text = {name: df.copy() for name, df in tables.items()}   # snapshot ไว้ verify ที่ 2.7

# ---- assumption ทั้งหมดของ PART นี้ รวมไว้ที่เดียว แก้แล้วรันใหม่ได้เลย ----
CONFIG = {
    # 'RT' (room temperature): ใน literature มักหมายถึง 20-25 °C — เลือก 25 เป็นค่าแทน
    "RT_CELSIUS": 25.0,
    # 1 Å = 0.1 nm
    "ANGSTROM_PER_NM": 10.0,
    # ช่วงค่า a-b ใช้ค่ากลาง (a+b)/2 เป็นตัวแทน (min/max ถูกเก็บไว้เสมอ)
    "RANGE_REPRESENTATIVE": "mid",
}

NUM = r"-?\d+(?:\.\d+)?(?:[eE][+-]?\d+)?"   # ตัวเลขทศนิยม (รองรับติดลบ + สัญกรณ์วิทยาศาสตร์ เช่น 4.9E+15)

def is_num(s) -> bool:
    return isinstance(s, str) and re.fullmatch(NUM, s.strip()) is not None

audit_rows = []                     # ทุกแถวที่ถูกตีความจะถูกบันทึกลงนี่
print("CONFIG:", CONFIG)

CONFIG: {'RT_CELSIUS': 25.0, 'ANGSTROM_PER_NM': 10.0, 'RANGE_REPRESENTATIVE': 'mid'}


## 2.1 Scan — คอลัมน์ตัวเลขที่ต้องตรวจ

In [23]:
NUM_COLS = {
    "layers":     ["thickness_nm"],
    "materials":  ["amount"],
    "properties": ["value", "error"],
    "stability":  ["temperature_c", "humidity_pct", "time_hr", "retention_pct",
                   "t80_hr", "initial_pce", "final_pce"],
    "vendors":    ["purity"],
}

scan = []
for name, cols in NUM_COLS.items():
    for col in cols:
        s = tables[name][col].dropna().str.strip()
        s = s[s != ""]
        bad = s[~s.map(is_num)]
        scan.append({"table": name, "column": col,
                     "n_filled": len(s), "n_bad": len(bad),
                     "bad_examples": ", ".join(sorted(bad.unique())[:5])})
scan = pd.DataFrame(scan).sort_values("n_bad", ascending=False).reset_index(drop=True)
print(f"แถวที่ parse ไม่ได้รวม: {scan.n_bad.sum()}")
scan

แถวที่ parse ไม่ได้รวม: 79


,table,column,n_filled,n_bad,bad_examples
0,vendors,purity,346,48,"20 wt, 9 - 15 Ω □-1, >97, >99.0, >99.99"
1,layers,thickness_nm,105,12,"1200 Å, 4, 100, 400 Å, 400-450, 90 Å"
2,stability,humidity_pct,26,12,"25 ± 5, 30-50, 45–55, 50 - 60, 50 ± 5"
3,stability,temperature_c,48,6,RT
4,stability,retention_pct,50,1,< 55
5,properties,error,74,0,
6,materials,amount,323,0,
7,properties,value,544,0,
8,stability,time_hr,59,0,
9,stability,t80_hr,0,0,


## 2.2 `layers.thickness_nm` — หน่วย Å, ช่วงค่า a-b, หลายชั้นย่อยในเซลล์เดียว

In [24]:
def parse_thickness(v):
    """คืน (parsed, vmin, vmax, note) — note ว่าง = ตัวเลขปกติไม่ต้องตีความ"""
    if not isinstance(v, str) or not v.strip():
        return (None, None, None, "")
    s = v.strip()
    if is_num(s):                                     # ตัวเลขปกติ
        x = float(s)
        return (x, x, x, "")
    m = re.fullmatch(rf"({NUM})\s*Å", s)              # หน่วย Å
    if m:
        x = float(m[1]) / CONFIG["ANGSTROM_PER_NM"]
        return (x, x, x, f"แปลงหน่วย: {s} = {x:g} nm (1 Å = 0.1 nm)")
    m = re.fullmatch(rf"({NUM})\s*[-–]\s*({NUM})", s)  # ช่วง a-b
    if m:
        a, b = float(m[1]), float(m[2])
        return ((a + b) / 2, a, b, f"ช่วงค่า: ใช้ค่ากลาง {(a + b) / 2:g} (min={a:g}, max={b:g})")
    m = re.fullmatch(rf"({NUM})\s*,\s*({NUM})", s)     # หลายชั้นย่อย a, b
    if m:
        a, b = float(m[1]), float(m[2])
        return (a + b, min(a, b), max(a, b),
                f"หลายชั้นย่อยในเซลล์เดียว: ผลรวม {a + b:g} nm (ชั้นย่อย {a:g} + {b:g})")
    return (None, None, None, f"parse ไม่ได้: {s!r}")

lay = tables["layers"]
parsed = lay["thickness_nm"].map(parse_thickness)
lay["thickness_nm_parsed"] = parsed.map(lambda t: t[0])
lay["thickness_nm_min"] = parsed.map(lambda t: t[1])
lay["thickness_nm_max"] = parsed.map(lambda t: t[2])
lay["thickness_nm_note"] = parsed.map(lambda t: t[3])

hit = lay["thickness_nm_note"] != ""
for _, r in lay[hit].iterrows():
    audit_rows.append({"table": "layers", "column": "thickness_nm",
                       "device_id": r["device_id"], "raw": r["thickness_nm"],
                       "parsed": r["thickness_nm_parsed"], "vmin": r["thickness_nm_min"],
                       "vmax": r["thickness_nm_max"], "extra": r["layer_type"],
                       "note": r["thickness_nm_note"]})

print(f"แถวที่ถูกตีความ: {int(hit.sum())}")
lay.loc[hit, ["device_id", "layer_type", "material", "thickness_nm",
              "thickness_nm_parsed", "thickness_nm_min", "thickness_nm_max",
              "thickness_nm_note"]].rename(
    columns={"thickness_nm": "before (raw)", "thickness_nm_parsed": "after (parsed)"})

แถวที่ถูกตีความ: 12


,device_id,layer_type,material,before (raw),after (parsed),thickness_nm_min,thickness_nm_max,thickness_nm_note
3,10.1002admi.202500231_Sample1,PSK,[ (Cs.05FA.79MA.16)Pb(I.87Br.13)3 ],400-450,425.0,400.0,450.0,"ช่วงค่า: ใช้ค่ากลาง 425 (min=400, max=450)"
11,10.1002admi.202500231_Sample2,PSK,[ (Cs.05FA.79MA.16)Pb(I.87Br.13)3 ],400-450,425.0,400.0,450.0,"ช่วงค่า: ใช้ค่ากลาง 425 (min=400, max=450)"
19,10.1002admi.202500231_Sample3,PSK,[ (Cs.05FA.79MA.16)Pb(I.87Br.13)3 ],400-450,425.0,400.0,450.0,"ช่วงค่า: ใช้ค่ากลาง 425 (min=400, max=450)"
27,10.1002admi.202500231_Sample4,PSK,[ (Cs.05FA.79MA.16)Pb(I.87Br.13)3 ],400-450,425.0,400.0,450.0,"ช่วงค่า: ใช้ค่ากลาง 425 (min=400, max=450)"
119,10_1002_solr_202200422_Sample1,Cathode,Cr + Au,"4, 100",104.0,4.0,100.0,หลายชั้นย่อยในเซลล์เดียว: ผลรวม 104 nm (ชั้นย่อย 4 + 100)
127,10_1002_solr_202200422_Sample2,Cathode,Cr + Au,"4, 100",104.0,4.0,100.0,หลายชั้นย่อยในเซลล์เดียว: ผลรวม 104 nm (ชั้นย่อย 4 + 100)
134,10_1002_solr_202400534_Sample1,ETL treatment (1),PCBM,400 Å,40.0,40.0,40.0,แปลงหน่วย: 400 Å = 40 nm (1 Å = 0.1 nm)
135,10_1002_solr_202400534_Sample1,ETL treatment (2),BCP,90 Å,9.0,9.0,9.0,แปลงหน่วย: 90 Å = 9 nm (1 Å = 0.1 nm)
136,10_1002_solr_202400534_Sample1,Cathode,Cu,1200 Å,120.0,120.0,120.0,แปลงหน่วย: 1200 Å = 120 nm (1 Å = 0.1 nm)
143,10_1002_solr_202400534_Sample2,ETL treatment (1),PCBM,400 Å,40.0,40.0,40.0,แปลงหน่วย: 400 Å = 40 nm (1 Å = 0.1 nm)


## 2.3 `stability` — RT, ช่วง/±, censored (`<x`/`>x`)

In [25]:
def parse_temperature(v):
    if not isinstance(v, str) or not v.strip():
        return (None, "")
    s = v.strip()
    if is_num(s):
        return (float(s), "")
    if s.upper() == "RT":
        return (CONFIG["RT_CELSIUS"],
                f"RT (room temperature) -> {CONFIG['RT_CELSIUS']:g} C [assumption: CONFIG]")
    return (None, f"parse ไม่ได้: {s!r}")

def parse_range_pm(v):
    """สำหรับ humidity: ตัวเลข | ช่วง a-b | c ± t -> (parsed, min, max, note)"""
    if not isinstance(v, str) or not v.strip():
        return (None, None, None, "")
    s = v.strip()
    if is_num(s):
        x = float(s)
        return (x, x, x, "")
    m = re.fullmatch(rf"({NUM})\s*[-–]\s*({NUM})", s)
    if m:
        a, b = float(m[1]), float(m[2])
        return ((a + b) / 2, a, b, f"ช่วงค่า: ใช้ค่ากลาง {(a + b) / 2:g} (min={a:g}, max={b:g})")
    m = re.fullmatch(rf"({NUM})\s*±\s*({NUM})", s)
    if m:
        c, t = float(m[1]), float(m[2])
        return (c, c - t, c + t, f"ค่ากลาง±คลาดเคลื่อน: {c:g} (min={c - t:g}, max={c + t:g})")
    return (None, None, None, f"parse ไม่ได้: {s!r}")

def parse_censored(v):
    """สำหรับ retention: ตัวเลข | '< x' -> (parsed, censor, note)"""
    if not isinstance(v, str) or not v.strip():
        return (None, "", "")
    s = v.strip()
    if is_num(s):
        return (float(s), "", "")
    m = re.fullmatch(rf"([<>])\s*({NUM})", s)
    if m:
        return (float(m[2]), m[1],
                f"ค่า censored: รู้แค่ว่า {'ต่ำ' if m[1] == '<' else 'สูง'}กว่า {m[2]} — ห้ามใช้เป็นค่าจริงโดยไม่ดูธง censor")
    return (None, "", f"parse ไม่ได้: {s!r}")

stb = tables["stability"]

t_ = stb["temperature_c"].map(parse_temperature)
stb["temperature_c_parsed"] = t_.map(lambda x: x[0])
stb["temperature_c_note"] = t_.map(lambda x: x[1])

h = stb["humidity_pct"].map(parse_range_pm)
stb["humidity_pct_parsed"] = h.map(lambda x: x[0])
stb["humidity_pct_min"] = h.map(lambda x: x[1])
stb["humidity_pct_max"] = h.map(lambda x: x[2])
stb["humidity_pct_note"] = h.map(lambda x: x[3])

r = stb["retention_pct"].map(parse_censored)
stb["retention_pct_parsed"] = r.map(lambda x: x[0])
stb["retention_pct_censor"] = r.map(lambda x: x[1])
stb["retention_pct_note"] = r.map(lambda x: x[2])

for col, pcol, note_col in [("temperature_c", "temperature_c_parsed", "temperature_c_note"),
                            ("humidity_pct", "humidity_pct_parsed", "humidity_pct_note"),
                            ("retention_pct", "retention_pct_parsed", "retention_pct_note")]:
    hit = stb[note_col] != ""
    for _, row in stb[hit].iterrows():
        audit_rows.append({"table": "stability", "column": col,
                           "device_id": row["device_id"], "raw": row[col],
                           "parsed": row[pcol],
                           "vmin": row.get("humidity_pct_min") if col == "humidity_pct" else None,
                           "vmax": row.get("humidity_pct_max") if col == "humidity_pct" else None,
                           "extra": row.get("retention_pct_censor", "") if col == "retention_pct" else "",
                           "note": row[note_col]})

hit_any = (stb["temperature_c_note"] != "") | (stb["humidity_pct_note"] != "") | (stb["retention_pct_note"] != "")
print(f"แถวที่ถูกตีความ: {int(hit_any.sum())}")
stb.loc[hit_any, ["device_id", "environment", "temperature_c", "temperature_c_parsed",
                  "humidity_pct", "humidity_pct_parsed", "humidity_pct_min", "humidity_pct_max",
                  "retention_pct", "retention_pct_parsed", "retention_pct_censor"]]

แถวที่ถูกตีความ: 18


,device_id,environment,temperature_c,temperature_c_parsed,humidity_pct,humidity_pct_parsed,humidity_pct_min,humidity_pct_max,retention_pct,retention_pct_parsed,retention_pct_censor
8,10.1016j.nanoen.2021.106285_Sample1,Thermal Aging,85,85.0,50 - 60,55.0,50.0,60.0,NaN,NaN,
9,10.1016j.nanoen.2021.106285_Sample2,Thermal Aging,85,85.0,50–60,55.0,50.0,60.0,NaN,NaN,
11,10.1016j.nanoen.2021.106285_Sample3,Ambient MPP,NaN,NaN,45–55,50.0,45.0,55.0,90.0,90.00,
12,10.1016j.nanoen.2021.106285_Sample3,Thermal Aging,85.0,85.0,50–60,55.0,50.0,60.0,NaN,NaN,
14,10.1021_acsami.5c16972_Sample1,ambient,25,25.0,30-50,40.0,30.0,50.0,38,38.00,
16,10.1021_acsami.5c16972_Sample2,ambient,25,25.0,30-50,40.0,30.0,50.0,70,70.00,
18,10.1021_acsami.5c16972_Sample3,ambient,25,25.0,30-50,40.0,30.0,50.0,85,85.00,
20,10_1002_solr_202200422_Sample1,Ambient air,25,25.0,60 ± 5,60.0,55.0,65.0,62,62.00,
35,10_1016_j_solener_2025_113286_Sample1,ambient,RT,25.0,35,35.0,35.0,35.0,< 55,55.00,<
36,10_1016_j_solener_2025_113286_Sample2,ambient,RT,25.0,35,35.0,35.0,35.0,80,80.00,


## 2.4 `vendors.purity` — จำแนก 5 ชนิด (percent / bounded / grade / wt% / misfiled)

In [26]:
def parse_purity(v):
    """คืน (pct, qualifier, kind, note)"""
    if not isinstance(v, str) or not v.strip() or v.strip() == "Not provided":
        return (None, "", "missing", "")
    s = v.strip()
    if is_num(s):
        return (float(s), "=", "percent", "")
    m = re.fullmatch(rf"([>≥])\s*({NUM})", s)
    if m:
        return (float(m[2]), m[1], "percent_lower_bound",
                f"qualifier {m[1]}: ความบริสุทธิ์อย่างน้อย {m[2]}%")
    if s.lower() == "anhydrous":
        return (None, "", "grade_descriptor",
                "เกรดตัวทำละลาย 'ปราศจากน้ำ' — ไม่ใช่ตัวเลขความบริสุทธิ์")
    m = re.fullmatch(rf"({NUM})\s*wt", s)
    if m:
        return (None, "", "concentration_wt_pct",
                f"ความเข้มข้น {m[1]} wt% ของ dispersion — ไม่ใช่ purity (ข้อมูลผิดชนิด)")
    if "Ω" in s or "Ω" in s or "□" in s:
        return (None, "", "sheet_resistance_misfiled",
                f"sheet resistance ของ substrate ({s}) ถูกกรอกในช่อง purity — Ω ในไฟล์คือ U+2126 OHM SIGN")
    return (None, "", "unrecognized", f"จำแนกไม่ได้: {s!r}")

ven = tables["vendors"]
pu = ven["purity"].map(parse_purity)
ven["purity_pct"] = pu.map(lambda x: x[0])
ven["purity_qualifier"] = pu.map(lambda x: x[1])
ven["purity_kind"] = pu.map(lambda x: x[2])
_purity_note = pu.map(lambda x: x[3])

hit = ven["purity"].notna() & (ven["purity"].str.strip() != "") & ~ven["purity"].fillna("").map(is_num)
for idx in ven.index[hit]:
    audit_rows.append({"table": "vendors", "column": "purity",
                       "device_id": ven.at[idx, "device_id"], "raw": ven.at[idx, "purity"],
                       "parsed": ven.at[idx, "purity_pct"], "vmin": None, "vmax": None,
                       "extra": ven.at[idx, "purity_kind"], "note": _purity_note[idx]})

print("จำแนกชนิดได้ดังนี้ (นับเฉพาะแถวที่มีค่า):")
display(ven.loc[ven["purity"].notna() & (ven["purity"].str.strip() != ""), "purity_kind"]
           .value_counts().rename("n_rows").to_frame())

print("แถวที่ไม่ใช่ percent ปกติ (ตัวอย่างการจำแนก):")
ven.loc[hit, ["device_id", "material", "purity", "purity_pct",
              "purity_qualifier", "purity_kind"]].drop_duplicates(
    subset=["material", "purity", "purity_kind"]).reset_index(drop=True).rename(
    columns={"purity": "before (raw)", "purity_pct": "after (pct)"})

จำแนกชนิดได้ดังนี้ (นับเฉพาะแถวที่มีค่า):


,n_rows
purity_kind,
percent,298
percent_lower_bound,18
missing,17
grade_descriptor,6
sheet_resistance_misfiled,4
concentration_wt_pct,3


แถวที่ไม่ใช่ percent ปกติ (ตัวอย่างการจำแนก):


,device_id,material,before (raw),after (pct),purity_qualifier,purity_kind
0,10.1002admi.202500231_Sample1,ITO coated glass,9 - 15 Ω □-1,NaN,,sheet_resistance_misfiled
1,10.1002admi.202500231_Sample1,MABr,>99.99,99.99,>,percent_lower_bound
2,10.1002admi.202500231_Sample1,NiNO3.6H2O,>97,97.00,>,percent_lower_bound
3,10.1002admi.202500231_Sample1,NaOH,>97,97.00,>,percent_lower_bound
4,10.1016j.nanoen.2021.106285_Sample1,Al2O3,20 wt,NaN,,concentration_wt_pct
5,10_1002_solr_202400534_Sample1,BCP,>99.0,99.00,>,percent_lower_bound
6,10_1002_solr_202400534_Sample1,CsI,≥99.99,99.99,≥,percent_lower_bound
7,10_1002_solr_202400534_Sample1,PEAI,≥99.5,99.50,≥,percent_lower_bound
8,10_1016_j_solener_2025_113286_Sample1,DMSO,Anhydrous,NaN,,grade_descriptor
9,10_1016_j_solener_2025_113286_Sample1,Chlorobenzene,Anhydrous,NaN,,grade_descriptor


## 2.5 Save audit

In [27]:
audit = pd.DataFrame(audit_rows,
                     columns=["table", "column", "device_id", "raw",
                              "parsed", "vmin", "vmax", "extra", "note"])

ba = (audit.groupby(["table", "column", "raw", "parsed", "vmin", "vmax", "extra"],
                    dropna=False)
           .size().rename("n_rows").reset_index()
           .rename(columns={"raw": "before", "parsed": "after",
                            "extra": "kind/flag"})
           .sort_values(["table", "column", "before"])
           .reset_index(drop=True))
print(f"คู่ before -> after ไม่ซ้ำ: {len(ba)} | ครอบคลุม {ba.n_rows.sum()} แถวที่ถูกตีความ")
ba

คู่ before -> after ไม่ซ้ำ: 23 | ครอบคลุม 79 แถวที่ถูกตีความ


,table,column,before,after,vmin,vmax,kind/flag,n_rows
0,layers,thickness_nm,1200 Å,120.00,120.0,120.0,Cathode,2
1,layers,thickness_nm,"4, 100",104.00,4.0,100.0,Cathode,2
2,layers,thickness_nm,400 Å,40.00,40.0,40.0,ETL treatment (1),2
3,layers,thickness_nm,400-450,425.00,400.0,450.0,PSK,4
4,layers,thickness_nm,90 Å,9.00,9.0,9.0,ETL treatment (2),2
5,stability,humidity_pct,25 ± 5,25.00,20.0,30.0,,2
6,stability,humidity_pct,30-50,40.00,30.0,50.0,,3
7,stability,humidity_pct,45–55,50.00,45.0,55.0,,1
8,stability,humidity_pct,50 - 60,55.00,50.0,60.0,,1
9,stability,humidity_pct,50 ± 5,50.00,45.0,55.0,,2


In [28]:
audit.to_csv(AUDIT / "value_parse_audit.csv", index=False)
print("บันทึก audit แล้ว:", AUDIT / "value_parse_audit.csv", f"({len(audit)} แถว)")
audit.groupby(["table", "column"]).size().rename("n_rows").to_frame()

บันทึก audit แล้ว: D:\University\Year 4 Semester 1 (Intern)\Perovskites-Research\processed\audit\value_parse_audit.csv (79 แถว)


n_rows
table     column               
layers    thickness_nm       12
stability humidity_pct       12
          retention_pct       1
          temperature_c       6
vendors   purity             48

## 2.6 Verify — 3 เงื่อนไขต้องผ่านครบ (เทียบกับ snapshot ก่อนเริ่ม PART 2)

In [29]:
raw_ok = all(tables[n][c].fillna("").equals(tables_after_text[n][c].fillna(""))
             for n in tables_after_text for c in tables_after_text[n].columns)
rows_ok = all(len(tables[n]) == len(tables_after_text[n]) for n in tables)

n_bad_total = 0
unhandled = []
for name, cols in NUM_COLS.items():
    for col in cols:
        s = tables[name][col].fillna("").str.strip()
        bad_idx = s[(s != "") & ~s.map(is_num)].index
        n_bad_total += len(bad_idx)
        for i in bad_idx:
            note_cols = [c for c in tables[name].columns
                         if c.startswith(col) and c != col]
            vals = [str(tables[name].at[i, c]) for c in note_cols]
            handled = any(v not in ("", "nan", "None") for v in vals)
            flagged_bad = any("parse ไม่ได้" in v or v == "unrecognized" for v in vals)
            if not handled or flagged_bad:
                unhandled.append((name, col, int(i), s[i]))

print("1) คอลัมน์เดิมไม่ถูกแก้ระหว่าง PART 2 :", raw_ok)
print("2) จำนวนแถวเท่าเดิมทุกตาราง            :", rows_ok)
print(f"3) เคส non-numeric {n_bad_total} แถว | ใน audit {len(audit)} แถว | หลุด: {len(unhandled)}")
assert raw_ok and rows_ok and n_bad_total == len(audit) and not unhandled, "VERIFY FAILED"
print("PART 2 verify ผ่านหมด ✓")

1) คอลัมน์เดิมไม่ถูกแก้ระหว่าง PART 2 : True
2) จำนวนแถวเท่าเดิมทุกตาราง            : True
3) เคส non-numeric 79 แถว | ใน audit 79 แถว | หลุด: 0
PART 2 verify ผ่านหมด ✓


## 2.7 เขียนออก `processed/clean/*.csv`

จุดส่งมอบเดียวของ `column_prepare.ipynb` — `eda_prepare.ipynb` PART 3 จะอ่านต่อจากที่นี่

In [30]:
DST = ROOT / "processed" / "clean"
DST.mkdir(parents=True, exist_ok=True)
for name, df in tables.items():
    df.to_csv(DST / f"{name}.csv", index=False)
print("wrote", len(tables), "tables ->", DST)
for name, df in tables.items():
    print(f"  {name:12s} {df.shape}")

wrote 7 tables -> D:\University\Year 4 Semester 1 (Intern)\Perovskites-Research\processed\clean
  devices      (44, 4)
  layers       (352, 9)
  materials    (543, 7)
  process      (702, 9)
  properties   (563, 6)
  stability    (62, 19)
  vendors      (644, 7)
